# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

In [ ]:
from src.utils.env import set_seed

set_seed(42)

In [ ]:
import torch
from scripts.utils.load_model import load_model

torch.set_float32_matmul_precision("high")

model, tokenizer = load_model("google/gemma-2b-it", torch_dtype="bfloat16")

In [ ]:
model

In [ ]:
from src.model import TargetedModel

targeted_model = TargetedModel(model, tokenizer, is_chat=True)

In [ ]:
from src.utils.logging import create_logger

logger = create_logger(__name__)

In [ ]:
from src.aliases import Conv

In [ ]:
from scripts.utils.load_dataset import load_dataset

ds_train, ds_val, ds_test = load_dataset("hh-rlhf")

In [ ]:
from src.utils.data import TableLoader

dl_train = TableLoader(ds_train, batch_size=8, shuffle=True)
dl_val = TableLoader(ds_val, batch_size=8, shuffle=False)
dl_test = TableLoader(ds_test, batch_size=8, shuffle=False)

In [ ]:
from src.functional import compute_targets_mask, project

In [ ]:
import matplotlib.pyplot as plt
from transformers.tokenization_utils import PreTrainedTokenizer
from transformers import PreTrainedModel
from src.activation_extractor import ActivationExtractor
import numpy as np


@torch.inference_mode()
def plot_conversation_norms(
    targeted_model: TargetedModel,
    conv: Conv | str,
    layer_name: str,
    direction: torch.Tensor,
    ax: plt.Axes | None = None,
    title_suffix: str = "",
):

    activ_extractor = ActivationExtractor(targeted_model.model, layer_name)
    encodings = targeted_model.tokenize(conv)

    with activ_extractor.capture():
        targeted_model.forward(encodings)
        activs = activ_extractor.get_activations()[layer_name]

    # compute normalized projections
    activs = torch.nn.functional.normalize(activs, dim=-1)
    projections = project(activs, direction=direction, normalize=True)
    l2_norms = torch.norm(projections, dim=-1)

    targets_mask = compute_targets_mask(encodings)
    token_list = encodings.input_ids[targets_mask.bool()].tolist()
    norms_list = l2_norms[targets_mask.bool()].tolist()
    token_strs = [targeted_model.tokenizer.decode([tid]) for tid in token_list]

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(15, 3))

    x_pos = np.arange(len(token_strs))
    ax.bar(x_pos, norms_list, alpha=0.7)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(token_strs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("L2 Norm")
    ax.set_title(f"Prompt: Projection Magnitudes {title_suffix}")
    ax.grid(axis="y", alpha=0.3)


# Multi-Location Redundant Subspace Search

In [ ]:
from math import floor


def get_search_locations(
    num_probes=10,
    block_path: str | None = "model.layers.{i}",
    attn_path: str | None = "model.layers.{i}.self_attn",
    mlp_path: str | None = "model.layers.{i}.mlp",
) -> list[tuple[str, str, str]]:
    """
    Define architecturally meaningful locations to search for redundant directions.
    """
    num_hidden_layers = model.config.num_hidden_layers
    num_probes = min(num_probes, num_hidden_layers)

    indices = [floor(i * num_hidden_layers / num_probes) for i in range(num_probes)]
    if num_hidden_layers - 1 not in indices:
        indices.append(num_hidden_layers - 1)

    locations = []

    if block_path is not None:
        locations += [(f"residual_{i}", block_path.format(i=i), f"Residual stream after block {i}") for i in indices]

    if attn_path is not None:
        locations += [(f"attn_output_{i}", attn_path.format(i=i), f"Attention output after block {i}") for i in indices]

    if mlp_path is not None:
        locations += [(f"mlp_output_{i}", mlp_path.format(i=i), f"MLP output after block {i}") for i in indices]

    special_locations = [
        ("embeddings", "model.embed_tokens", "Token embeddings (before any layers)"),
        ("final_norm", "model.norm", "Final normalization (before lm_head)"),
    ]

    locations += special_locations

    print("=" * 80)
    print(f"REDUNDANCY EXPLORATION: {len(locations)} LOCATIONS")
    print("=" * 80)
    print(f"\n{'Name':<30} {'Path':<50} Description")
    print("=" * 80)
    for name, path, desc in locations:
        print(f"  {name:<30} {path:<50} {desc}")

    # return locations
    return [locations[0]]


# Get search locations
search_locations = get_search_locations()

In [ ]:
from src.norm.train import train_norm
from src.norm.evaluate import evaluate_norm

from src.utils.trackers import MetricTracker
import time


def search_redundant_directions_multi_location(
    targeted_model: TargetedModel,
    search_locations: list[tuple[str, str, str]],
    dl_train: TableLoader,
    dl_val: TableLoader,
    num_steps: int = 500,
    lr: float = 0.01,
    proj_weight: float = 0.1,
) -> dict[str, dict]:
    """
    Search for redundant 1D subspaces across multiple network locations.
    """
    location_results = {}

    print("=" * 80)
    print("SEARCHING FOR REDUNDANT 1D SUBSPACES ACROSS MULTIPLE LOCATIONS")
    print("=" * 80)
    print(f"  Training set:   {dl_train.n_samples} conversations")
    print(f"  Validation set: {dl_val.n_samples} conversations")

    for location_name, layer_path, description in search_locations:
        print(f"\n{'=' * 80}")
        print(f"LOCATION: {location_name}")
        print(f"Path: {layer_path}")
        print(f"Description: {description}")
        print(f"{'=' * 80}\n")

        start_time = time.time()

        metric_tracker = MetricTracker.create("test", kind="wandb", project="Silent-Direction", disabled=False)

        direction, history = train_norm(
            targeted_model=targeted_model,
            layer=layer_path,
            dl_train=dl_train,
            num_steps=num_steps,
            learning_rate=lr,
            proj_weight=proj_weight,
        )

        for step, metrics in enumerate(history):
            metric_tracker.report_scalars(metrics, step=step)

        val_metrics = evaluate_norm(
            targeted_model=targeted_model,
            layer=layer_path,
            direction=direction,
            dl_eval=dl_val,
        )

        elapsed = time.time() - start_time

        # Store results
        location_results[location_name] = {
            "layer_path": layer_path,
            "description": description,
            "direction": direction,
            "metrics": val_metrics,
            "time_sec": elapsed,
        }

        print(f"\n✓ Completed in {elapsed:.1f}s")
        metric_names = sorted(val_metrics.keys())
        for metric_name in metric_names:
            metric_value = val_metrics[metric_name]
            print(f"  {metric_name.replace('_', ' ').title():.<30} {metric_value:.6f}")

    print(f"\n{'=' * 80}")
    print("ALL LOCATIONS SEARCHED")
    print(f"{'=' * 80}")

    return location_results


In [ ]:
# Run the search across all locations
location_results = search_redundant_directions_multi_location(
    targeted_model=targeted_model,
    search_locations=search_locations,
    dl_train=dl_train,
    dl_val=dl_val,
    num_steps=500,
    lr=0.01,
    proj_weight=0.1,
)

In [ ]:
import pandas as pd


def _create_comparison_dataframe(location_results: dict[str, dict]) -> pd.DataFrame:
    """Build comparison dataframe from location results."""
    data = []
    for name, result in location_results.items():
        m = result["metrics"]

        res = {
            "Location": name,
            "Description": result["description"],
        }

        for metric_name, metric_value in m.items():
            res[metric_name.replace("_", " ").title()] = f"{metric_value:.6f}"

        data.append(res)

    return pd.DataFrame(data)


def analyze_and_compare_results(location_results: dict[str, dict]) -> pd.DataFrame:
    print(f"\n{'=' * 120}\nCOMPARISON: REDUNDANT SUBSPACES ACROSS LOCATIONS\n{'=' * 120}\n")

    df_comparison = _create_comparison_dataframe(location_results)
    print(df_comparison.to_string(index=False))

    print(f"\n{'=' * 120}\nSUMMARY STATISTICS\n{'=' * 120}")

    # find numeric columns in the dataframe
    numeric_columns = [col for col in df_comparison.columns if col not in ["Location", "Description"]]

    for col in numeric_columns:
        # convert column to float for analysis
        values = df_comparison[col].astype(float)

        mean_val = values.mean()
        std_val = values.std()
        min_val = values.min()
        max_val = values.max()
        median_val = values.median()

        print(f"\n{col}:")
        print(f"  Mean: {mean_val:.6f}")
        print(f"  Std:  {std_val:.6f}")
        print(f"  Min:  {min_val:.6f} (at {df_comparison.loc[values.idxmin(), 'Location']})")
        print(f"  Max:  {max_val:.6f} (at {df_comparison.loc[values.idxmax(), 'Location']})")
        print(f"  Median: {median_val:.6f}")

    print(f"\n{'=' * 120}\nKEY INSIGHTS\n{'=' * 120}")

    for col in numeric_columns:
        values = df_comparison[col].astype(float)

        max_row = df_comparison.loc[values.idxmax()]
        min_row = df_comparison.loc[values.idxmin()]

        print(f"\n{col}:")
        print(f"  • HIGHEST: {max_row['Location']} ({max_row[col]:.6f})")
        print(f"    Description: {max_row['Description']}")
        print(f"  • LOWEST:  {min_row['Location']} ({min_row[col]:.6f})")
        print(f"    Description: {min_row['Description']}")

    return df_comparison


df_comparison = analyze_and_compare_results(location_results)


In [ ]:
import random
import matplotlib.pyplot as plt


@torch.inference_mode()
def visualize_top_locations(
    location_results: dict[str, dict],
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    convs: list[Conv],
    sort_metric: str = "redundancy_score",
    top_n: int = 3,
    num_vis_samples: int = 10,
) -> None:
    """
    Visualize projection patterns for top N locations by specified metric
    """
    sorted_locations = sorted(location_results.items(), key=lambda x: x[1]["metrics"][sort_metric], reverse=True)[:top_n]

    num_convs = min(len(convs), num_vis_samples)
    convs_sample = random.sample(convs, num_convs)

    for rank, (location_name, result) in enumerate(sorted_locations, 1):
        print(f"\n{'-' * 80}\nRANK {rank}: {location_name}\n{'-' * 80}")
        print(f"{result['description']}")
        print(f"{sort_metric.replace('_', ' ').title()}: {result['metrics'][sort_metric]:.4f}\n")

        layer_path = result["layer_path"]
        redundant_dir = result["direction"]

        fig, axes = plt.subplots(nrows=num_convs, ncols=1, figsize=(15, 3 * num_convs))

        for i, conv in enumerate(convs_sample):
            plot_conversation_norms(
                model=model,
                tokenizer=tokenizer,
                conv=conv,
                layer_name=layer_path,
                direction=redundant_dir,
                title_suffix=f"[{location_name}]",
                ax=axes[i],
            )


visualize_top_locations(
    location_results=location_results,
    model=model,
    tokenizer=tokenizer,
    convs=dl_val,
    sort_metric="redundancy_score",
    top_n=3,
    num_vis_samples=5,
)


# Long-Term Generation Test

Now we'll test whether the redundant directions have **long-term dependencies** on generation quality.

Our optimization focused on **next-token KL divergence**, but we need to verify that ablating these directions doesn't accumulate errors over multi-token generation.

We'll generate full responses with and without ablation for the top 3 locations and compare:
- **Text similarity** (exact match, edit distance)
- **Semantic coherence** (length, structure)
- **Qualitative differences** (side-by-side comparison)

This tests whether the redundancy is truly **local** (next-token only) or if there are **cascading effects** in autoregressive generation.

In [ ]:
import difflib
from src.model_helpers import generate, generate_ablated


@torch.inference_mode()
def test_generation_with_ablation(
    targeted_model: TargetedModel,
    sorted_locations: list[tuple[str, dict]],
    test_convs: list[Conv],
    max_new_tokens: int = 100,
) -> None:
    """
    Test long-term generation with and without direction ablation at top redundant locations.
    Prints outputs for manual semantic analysis.
    """
    print("\n" + "=" * 80)
    print("LONG-TERM GENERATION TEST: TOP REDUNDANT LOCATIONS")
    print("=" * 80)
    print("Note: Focus on semantic similarity rather than exact word matching.")
    print("Differences are highlighted for easier comparison.")

    for rank, (location_name, result) in enumerate(sorted_locations, 1):
        print(f"\n{'=' * 80}")
        print(f"TESTING LOCATION {rank}: {location_name}")
        print(f"Description: {result['description']}")
        print(f"{'=' * 80}\n")

        layer_path = result["layer_path"]
        redundant_dir = result["direction"]

        # Generate baseline (no ablation)
        print("Generating baseline outputs (NO ablation)...")
        baseline_outputs = generate(
            model=model,
            tokenizer=tokenizer,
            prompts=test_convs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        # Generate with ablation
        print("Generating ablated outputs (WITH ablation)...")
        ablated_outputs = generate_ablated(
            model=model,
            tokenizer=tokenizer,
            layer_name=layer_path,
            direction=redundant_dir,
            prompts=test_convs,
            max_new_tokens=max_new_tokens,
        )

        # Print comparisons for each prompt
        for i, conv in enumerate(test_convs):
            print(f"\nPrompt {i + 1}:")
            user_message = conv[0]["content"]
            print(f"User: {user_message}\n")

            baseline_response = baseline_outputs[i].strip()
            ablated_response = ablated_outputs[i].strip()

            print("BASELINE RESPONSE:")
            print(baseline_response)
            print("\nABLATED RESPONSE:")
            print(ablated_response)

            # Show differences if any
            if baseline_response != ablated_response:
                print("\nDIFFERENCES (unified diff):")
                diff = list(
                    difflib.unified_diff(
                        baseline_response.splitlines(keepends=True),
                        ablated_response.splitlines(keepends=True),
                        fromfile="Baseline",
                        tofile="Ablated",
                        lineterm="",
                    )
                )
                if diff:
                    for line in diff:
                        if line.startswith("+"):
                            print(f"\033[92m{line}\033[0m", end="")  # Green for additions
                        elif line.startswith("-"):
                            print(f"\033[91m{line}\033[0m", end="")  # Red for deletions
                        elif line.startswith("@@"):
                            print(f"\033[94m{line}\033[0m", end="")  # Blue for headers
                        else:
                            print(line, end="")
                else:
                    print("No line differences, but responses differ (possibly whitespace).")
            else:
                print("\nRESPONSES ARE IDENTICAL")

            print("\n" + "-" * 80)


In [ ]:
# Test generation on top 3 locations

sort_metric = "redundancy_score"  # Specify the metric to use for ranking locations

sorted_locations = [(k, res) for k, res in location_results.items()]
sorted_locations = sorted(sorted_locations, key=lambda item: item[1]["metrics"][sort_metric], reverse=True)[:3]

generation_results = test_generation_with_ablation(
    model=model,
    tokenizer=tokenizer,
    sorted_locations=sorted_locations,
    test_convs=test_convs[:10],
    max_new_tokens=100,
)